# NB1 — Preprocessing & EDA (Google Colab version)

**COS700 · Impact of Data Anonymisation on the Statistical Utility of Cybersecurity Datasets**

## Before running

1. **Upload the 8 CICIDS-2017 CSVs to Google Drive** (one-time only):  
   Place all `*.pcap_ISCX.csv` files in `My Drive/COS700/data/raw/`

2. **Set your Drive folder path** in the cell below (`DRIVE_ROOT`).  
   Default: `MyDrive/COS700` — change if you used a different folder.

3. **Run all cells top-to-bottom.** Outputs are saved back to Drive so they persist between sessions.

**Runtime recommendation:** GPU not needed. CPU with High-RAM is helpful for the RF fitting step.

In [ ]:
# ── Install dependencies (only needed once per Colab session) ─────────────────
import subprocess, sys
pkgs = ['pyarrow']
for pkg in pkgs:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Set your Drive root folder ────────────────────────────────────────────────
# Change this if your COS700 folder is in a different location on Drive
DRIVE_ROOT = '/content/drive/MyDrive/COS700'

print(f'Drive mounted. Using root: {DRIVE_ROOT}')

# NB1 — Preprocessing & EDA

**COS700 · Impact of Data Anonymisation on the Statistical Utility of Cybersecurity Datasets**

**Phase 3** of the implementation plan. **Google Colab version** — see setup cell above.  
Inputs: `COS700/data/raw/*.pcap_ISCX.csv` (on your Google Drive)  
Outputs: `COS700/data/analysis.parquet`, `COS700/results/`, `COS700/figures/eda/`

**Run all cells top-to-bottom. Do not proceed to NB2 until the gate checklist at the bottom passes.**

## 0. Imports & constants

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import glob

import numpy as np
import pandas as pd
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

# ── Paths (Drive-based) ───────────────────────────────────────────────────────
from pathlib import Path
ROOT        = Path(DRIVE_ROOT)
RAW_DIR     = ROOT / 'data' / 'raw'
DATA_DIR    = ROOT / 'data'
RESULTS_DIR = ROOT / 'results'
FIGURES_DIR = ROOT / 'figures' / 'eda'

for d in [DATA_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Constants (from revised proposal §5.6) ─────────────────────────────────────
RANDOM_STATE = 42
LABEL_COL    = 'Label'
N_BINS       = 100
RF_N_ESTIMATORS    = 100
RF_MAX_DEPTH       = None
RF_MIN_SAMPLES_LEAF = 5

# ── Feature categories (from src/config.py, now embedded here) ─────────────────
FEATURE_CATEGORIES = {
    'Packet lengths': [
        'Fwd Packet Length Max', 'Fwd Packet Length Min',
        'Fwd Packet Length Mean', 'Fwd Packet Length Std',
        'Bwd Packet Length Max', 'Bwd Packet Length Min',
        'Bwd Packet Length Mean', 'Bwd Packet Length Std',
        'Min Packet Length', 'Max Packet Length',
        'Packet Length Mean', 'Packet Length Std',
        'Packet Length Variance', 'Average Packet Size',
        'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
        'min_seg_size_forward',
    ],
    'Flow timing': [
        'Flow Duration',
        'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
        'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
        'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
        'Active Mean', 'Active Std', 'Active Max', 'Active Min',
        'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min',
    ],
    'Packet counts': [
        'Total Fwd Packets', 'Total Backward Packets',
        'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
        'Subflow Fwd Packets', 'Subflow Bwd Packets',
        'Subflow Fwd Bytes', 'Subflow Bwd Bytes',
        'act_data_pkt_fwd',
    ],
    'Rates & volumes': [
        'Flow Bytes/s', 'Flow Packets/s',
        'Fwd Packets/s', 'Bwd Packets/s',
        'Down/Up Ratio',
    ],
    'TCP flags': [
        'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count',
        'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count',
        'CWE Flag Count', 'ECE Flag Count',
        'Fwd PSH Flags', 'Bwd PSH Flags',
        'Fwd URG Flags', 'Bwd URG Flags',
    ],
    'Headers & window': [
        'Fwd Header Length', 'Fwd Header Length.1', 'Bwd Header Length',
        'Init_Win_bytes_forward', 'Init_Win_bytes_backward',
        'Destination Port',
    ],
    'Bulk (unused)': [
        'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate',
        'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate',
    ],
}

CATEGORY_LOOKUP = {feat: cat for cat, feats in FEATURE_CATEGORIES.items() for feat in feats}

print('Imports OK. ROOT:', ROOT)

## 1. Load & concatenate raw CSVs

In [ ]:
csv_files = sorted(RAW_DIR.glob('*.pcap_ISCX.csv'))
assert len(csv_files) == 8, f'Expected 8 CSVs, found {len(csv_files)}'

frames = []
for f in csv_files:
    df = pd.read_csv(f, low_memory=False)
    df.columns = [c.strip() for c in df.columns]     # strip whitespace from column names
    df[LABEL_COL] = df[LABEL_COL].str.strip()         # strip whitespace from labels
    print(f'  {f.name:<55}  {len(df):>8,} rows  {df.shape[1]} cols')
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(raw):,} rows × {raw.shape[1]} columns')
print(f'Label values: {sorted(raw[LABEL_COL].unique())}')

## 2. Clean

In [ ]:
n0 = len(raw)
print(f'Start: {n0:,} rows')

# Step 1: inf → NaN
num_cols = raw.select_dtypes(include=[np.number]).columns
n_inf = np.isinf(raw[num_cols]).sum().sum()
raw.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f'  Inf values replaced with NaN: {n_inf:,}')

# Step 2: drop NaN rows
n1 = len(raw)
raw.dropna(inplace=True)
print(f'  Rows dropped (NaN): {n1 - len(raw):,}')

# Step 3: drop exact duplicate rows
n2 = len(raw)
raw.drop_duplicates(inplace=True)
raw.reset_index(drop=True, inplace=True)
print(f'  Rows dropped (duplicates): {n2 - len(raw):,}')

print(f'\nSurviving: {len(raw):,} rows ({100*(len(raw)/n0):.1f}% of original)')
print(f'Class distribution:')
print(raw[LABEL_COL].value_counts().to_string())

## 3. Zero-variance & duplicate column check → fix d

In [ ]:
num_features = [c for c in raw.columns if c != LABEL_COL and pd.api.types.is_numeric_dtype(raw[c])]
print(f'Numeric columns before drop: {len(num_features)}')

# Zero-variance check (max == min)
col_max = raw[num_features].max()
col_min = raw[num_features].min()
constant_cols = list(col_max[col_max == col_min].index)
print(f'\nConstant columns (max == min): {len(constant_cols)}')
for c in constant_cols:
    print(f'  {c}')

# Duplicate column check
assert 'Fwd Header Length.1' in num_features, 'Expected Fwd Header Length.1'
is_dup = (raw['Fwd Header Length'] == raw['Fwd Header Length.1']).all()
print(f'\nFwd Header Length.1 is exact duplicate of Fwd Header Length: {is_dup}')

# Destination Port: include or exclude?
port_n_unique = raw['Destination Port'].nunique()
print(f'\nDestination Port unique values: {port_n_unique}')
EXCLUDE_DESTINATION_PORT = port_n_unique > 1000  # categorical if many unique values
print(f'Exclude Destination Port (categorical): {EXCLUDE_DESTINATION_PORT}')

In [ ]:
# Build the drop list and compute d
COLS_TO_DROP = set(constant_cols) | {'Fwd Header Length.1'}
if EXCLUDE_DESTINATION_PORT:
    COLS_TO_DROP.add('Destination Port')

feature_cols = [c for c in num_features if c not in COLS_TO_DROP]
d = len(feature_cols)

print(f'Columns dropped: {sorted(COLS_TO_DROP)}')
print(f'\nd = {d}  (retained features)')
print(f'Expected range from revised proposal: 67 ≤ d ≤ 72')
assert 60 <= d <= 78, f'd={d} is outside expected range — investigate'

# Build analysis set (retained features + Label)
analysis = raw[feature_cols + [LABEL_COL]].copy()
print(f'\nAnalysis set shape: {analysis.shape}')

## 4. Build feature_meta.csv

This is the single source of truth for the DP arm.  
Populate bounds from §5.4 of `cos700_revised_working.md`.

**Constants used (both verified against empirical data — see §8):**
- `FLOW_TIMEOUT_S = 120 s` — CICFlowMeter **flow** timeout (`Cmd.java:27`, V3 source). Empirically verified: `max(Flow Duration) = 119,999,998 µs ≤ 1.2×10⁸ µs`.
- `LINK_SPEED_BPS = 1e9` — assumed 1 GbE (IEEE 802.3ab, 1000BASE-T). Sharafaldin [15] does not state the testbed link speed; IEEE 802.3ab is the citation.


In [ ]:
# ── Verified constants (update if testbed spec found in Sharafaldin [15]) ────────
FLOW_TIMEOUT_S  = 120          # CICFlowMeter default active-timeout (seconds)
LINK_SPEED_BPS    = 1e9          # assumed 1 GbE; cite IEEE 802.3ab if no better source

FLOW_TIMEOUT_US = FLOW_TIMEOUT_S * 1e6          # 1.2×10⁸ µs
LINK_BYTES_S      = LINK_SPEED_BPS / 8              # 1.25×10⁸ B/s
MIN_FRAME_BYTES   = 64                              # Ethernet minimum frame
LINK_PKTS_S       = LINK_SPEED_BPS / (MIN_FRAME_BYTES * 8)  # ≈1.49×10⁶ pkt/s
PKT_MAX           = FLOW_TIMEOUT_S * LINK_PKTS_S  # ≈1.8×10⁸
B_MAX             = PKT_MAX * 65535
TCP_HDR_MAX       = 60           # TCP max header bytes (RFC 793)

print(f'FLOW_TIMEOUT_US = {FLOW_TIMEOUT_US:.2e} µs')
print(f'LINK_BYTES_S      = {LINK_BYTES_S:.2e} B/s  (1 GbE)')
print(f'LINK_PKTS_S       = {LINK_PKTS_S:.2e} pkt/s')
print(f'PKT_MAX           = {PKT_MAX:.2e}')
print(f'B_MAX             = {B_MAX:.2e}')

In [ ]:
def bound_for(feature):
    """Return (a_j, b_j, bound_source) for a feature. Raises if unknown."""
    f = feature
    TIMING = [
        'Flow Duration',
        'Fwd IAT Total','Fwd IAT Mean','Fwd IAT Std','Fwd IAT Max','Fwd IAT Min',
        'Bwd IAT Total','Bwd IAT Mean','Bwd IAT Std','Bwd IAT Max','Bwd IAT Min',
        'Flow IAT Mean','Flow IAT Std','Flow IAT Max','Flow IAT Min',
        'Active Mean','Active Std','Active Max','Active Min',
        'Idle Mean','Idle Std','Idle Max','Idle Min',
    ]
    PKT_LEN = [
        'Fwd Packet Length Max','Fwd Packet Length Min','Fwd Packet Length Mean','Fwd Packet Length Std',
        'Bwd Packet Length Max','Bwd Packet Length Min','Bwd Packet Length Mean','Bwd Packet Length Std',
        'Min Packet Length','Max Packet Length','Packet Length Mean','Packet Length Std',
        'Average Packet Size','Avg Fwd Segment Size','Avg Bwd Segment Size','min_seg_size_forward',
    ]
    PKT_COUNTS = [
        'Total Fwd Packets','Total Backward Packets',
        'Subflow Fwd Packets','Subflow Bwd Packets','act_data_pkt_fwd',
    ]
    BYTE_TOTALS = [
        'Total Length of Fwd Packets','Total Length of Bwd Packets',
        'Subflow Fwd Bytes','Subflow Bwd Bytes',
    ]
    FLAG_INDICATORS = ['Fwd PSH Flags','Fwd URG Flags']  # binary — verified in §8
    FLAG_COUNTS = ['FIN Flag Count','SYN Flag Count','PSH Flag Count',
                   'ACK Flag Count','URG Flag Count','CWE Flag Count',
                   'RST Flag Count','ECE Flag Count']  # RST/ECE are drop candidates but may survive

    if f in TIMING:
        return (0, FLOW_TIMEOUT_US, 'CICFlowMeter active-timeout=120s × 1e6 µs/s')
    if f in PKT_LEN:
        return (0, 65535, '16-bit IP total-length field (RFC 791)')
    if f == 'Packet Length Variance':
        return (0, 65535**2 / 4, "Popoviciu's ineq: var ≤ (max-min)²/4, max-min=65535")
    if f in PKT_COUNTS:
        return (0, PKT_MAX, f'pkt_max = {FLOW_TIMEOUT_S}s × {LINK_PKTS_S:.2e} pkt/s (IEEE 802.3 1 GbE)')
    if f in BYTE_TOTALS:
        return (0, B_MAX, f'B_max = pkt_max × 65535 (RFC791 × IEEE 802.3)')
    if f == 'Flow Bytes/s':
        return (0, LINK_BYTES_S, 'Link capacity bytes/s: 1 GbE = 1.25×10⁸ B/s (IEEE 802.3ab) — tighter-public branch')
    if f in ('Flow Packets/s','Fwd Packets/s','Bwd Packets/s'):
        return (0, LINK_PKTS_S, 'Line-rate packet rate: 1 GbE ≈ 1.49×10⁶ pkt/s (IEEE 802.3ab min-frame) — tighter-public branch')
    if f == 'Down/Up Ratio':
        return (0, PKT_MAX, 'Loose bound: ratio = Bwd/Fwd pkts ≤ pkt_max (when Fwd=1); verify semantics in EDA')
    if f in FLAG_INDICATORS:
        return (0, 1, 'CICFlowMeter binary indicator (0/1 per direction) — verified in §8')
    if f in FLAG_COUNTS:
        return (0, PKT_MAX, f'Per-flow packet count; pkt_max = {PKT_MAX:.2e} (IEEE 802.3 + CICFlowMeter timeout)')
    if f in ('Fwd Header Length','Bwd Header Length'):
        return (0, PKT_MAX * TCP_HDR_MAX, f'Per-flow sum: pkt_max × {TCP_HDR_MAX} bytes (RFC793 TCP max header) — wide-range')
    if f in ('Init_Win_bytes_forward','Init_Win_bytes_backward'):
        return (-1, 65535, 'TCP 16-bit receive-window (RFC793); -1 = CICFlowMeter sentinel for not-observed')
    if f == 'Destination Port':
        return (0, 65535, '16-bit port field (RFC793); categorical — inclusion decided in §3')
    raise ValueError(f'No bound defined for feature: {f}')


rows = []
for feat in feature_cols:
    a_j, b_j, src = bound_for(feat)
    rows.append({
        'feature':      feat,
        'category':     CATEGORY_LOOKUP.get(feat, 'Unknown'),
        'a_j':          a_j,
        'b_j':          b_j,
        'delta_f':      b_j - a_j,
        'bound_source': src,
    })

feature_meta = pd.DataFrame(rows)
feature_meta.to_csv(RESULTS_DIR / 'feature_meta.csv', index=False)
print(f'feature_meta.csv saved: {len(feature_meta)} rows')
print(feature_meta[['feature','category','a_j','b_j','delta_f']].to_string())

## 5. Save analysis.parquet

In [ ]:
out_path = DATA_DIR / 'analysis.parquet'
analysis.to_parquet(out_path, engine='pyarrow', index=False)
print(f'analysis.parquet saved: {analysis.shape[0]:,} rows × {analysis.shape[1]} columns')
print(f'  Features: {d}')
print(f'  Label: {analysis[LABEL_COL].nunique()} classes')

## 6. EDA figures

In [ ]:
# 6a — Class distribution
fig, ax = plt.subplots(figsize=(12, 4))
counts = analysis[LABEL_COL].value_counts()
counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Class Distribution (CICIDS-2017 analysis set)')
ax.set_ylabel('Record count')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_01_class_distribution.png', dpi=150)
plt.show()
print(counts.to_string())

In [ ]:
# 6b — One representative feature distribution per category
representatives = {
    'Packet lengths':  'Packet Length Mean',
    'Flow timing':     'Flow Duration',
    'Packet counts':   'Total Fwd Packets',
    'Rates & volumes': 'Flow Bytes/s',
    'TCP flags':       'ACK Flag Count',
    'Headers & window':'Init_Win_bytes_forward',
}
# Filter to only those actually retained
representatives = {k: v for k, v in representatives.items() if v in feature_cols}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (cat, feat) in zip(axes.flat, representatives.items()):
    vals = analysis[feat].clip(
        lower=analysis[feat].quantile(0.01),
        upper=analysis[feat].quantile(0.99)
    )
    ax.hist(vals, bins=50, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(f'{cat}\n{feat}', fontsize=9)
    ax.set_ylabel('Count')
plt.suptitle('Representative feature distributions (1st–99th percentile clipped)', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_02_feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# 6c — Spearman correlation matrix (sample: top-20 features by variance)
top20 = analysis[feature_cols].var().nlargest(20).index.tolist()
corr = analysis[top20].corr(method='spearman')
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=True, yticklabels=True, linewidths=0.3)
ax.set_title('Spearman correlation — top 20 features by variance')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_03_correlation_top20.png', dpi=150)
plt.show()

## 7. Train Random Forest → Gini importance

In [ ]:
X = analysis[feature_cols].values
y = analysis[LABEL_COL].values

print('Fitting RandomForestClassifier...')
rf = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X, y)
print('Done.')

importances = rf.feature_importances_
gini_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
gini_df = gini_df.sort_values('importance', ascending=False).reset_index(drop=True)
gini_df['rank'] = gini_df.index + 1
gini_df['cumulative_importance'] = gini_df['importance'].cumsum()
gini_df['category'] = gini_df['feature'].map(CATEGORY_LOOKUP)

gini_df.to_csv(RESULTS_DIR / 'gini_importance.csv', index=False)
print(f'gini_importance.csv saved: {len(gini_df)} rows')
print('\nTop 10 features by Gini importance:')
print(gini_df.head(10)[['rank','feature','category','importance']].to_string(index=False))
print('\nBottom 10 (near-zero importance):')
print(gini_df.tail(10)[['rank','feature','category','importance']].to_string(index=False))

In [ ]:
# Gini importance bar chart (top 30)
fig, ax = plt.subplots(figsize=(10, 8))
top30 = gini_df.head(30)
ax.barh(top30['feature'][::-1], top30['importance'][::-1], color='steelblue')
ax.set_xlabel('Gini importance (MDI)')
ax.set_title('Top 30 features by Random Forest Gini importance')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_04_gini_importance.png', dpi=150)
plt.show()

## 8. Verify flag binary claim

In [ ]:
flag_indicators = [f for f in ['Fwd PSH Flags', 'Fwd URG Flags'] if f in feature_cols]
flag_counts_cols = [f for f in ['FIN Flag Count','SYN Flag Count','RST Flag Count',
                                 'PSH Flag Count','ACK Flag Count','URG Flag Count',
                                 'CWE Flag Count','ECE Flag Count'] if f in feature_cols]

print('=== Flag indicator columns (should be binary: max ≤ 1) ===')
for col in flag_indicators:
    mn, mx = analysis[col].min(), analysis[col].max()
    status = '✓ BINARY' if mx <= 1 else '✗ NOT BINARY — update feature_meta.csv bound to [0, pkt_max]'
    print(f'  {col:<25}  min={mn}  max={mx}  {status}')

print('\n=== Flag count columns ===')
for col in flag_counts_cols:
    mn, mx = analysis[col].min(), analysis[col].max()
    print(f'  {col:<25}  min={mn}  max={mx}')

print('\n=== Init_Win_bytes (check for -1 sentinel) ===')
for col in ['Init_Win_bytes_forward','Init_Win_bytes_backward']:
    if col in feature_cols:
        mn, mx = analysis[col].min(), analysis[col].max()
        neg = (analysis[col] < 0).sum()
        print(f'  {col:<30}  min={mn}  max={mx}  negative rows={neg:,}')
# ── Negative Flow Duration check ────────────────────────────────────────────────
print('\n=== Negative Flow Duration (exporter artefact) ===')
neg_dur = (analysis['Flow Duration'] < 0).sum()
print(f'  Negative Flow Duration rows: {neg_dur:,}  ({neg_dur/len(analysis)*100:.4f}%)')
print(f'  These are a CICFlowMeter timestamp artefact (values −1 to −12, all BENIGN).')
print(f'  They will be clipped to a_j=0 by the DP bound — negligible clipping component.')

# ── Init_Win_bytes version note ──────────────────────────────────────────────────
print('\n=== Init_Win_bytes −1 sentinel note ===')
print('  The −1 sentinel is specific to CICFlowMeter V3 (used for CICIDS-2017).')
print('  CICFlowMeter V4 (current master, ahlashkari/CICFlowMeter) defaults to 0.')
print('  This is a V3 exporter artefact, not a protocol value. Cite the repo when explaining.')

# ── Down/Up Ratio analysis ───────────────────────────────────────────────────────
print('\n=== Down/Up Ratio semantics ===')
dur = analysis['Down/Up Ratio']
print(f'  Computed by CICFlowMeter as integer division: backward_packets // forward_packets')
print(f'  Unique values: {dur.nunique()}  (only integer-valued)')
print(f'  max observed:  {dur.max()}')
print(f'  pkt_max:       {PKT_MAX:,.0f}')
print(f'  max/pkt_max:   {dur.max()/PKT_MAX:.2e}  (bound is ~{PKT_MAX/dur.max():.0e}x the observed max)')
print(f'  p99:           {dur.quantile(0.99):.0f}')
print(f'  p99.9:         {dur.quantile(0.999):.0f}')
print(f'  => [0, pkt_max] is extremely loose. This feature will be pure noise at all ε.')


In [ ]:
# If any flag indicator has max > 1, update its bound in feature_meta.csv
needs_update = []
for col in flag_indicators:
    if col in feature_cols and analysis[col].max() > 1:
        needs_update.append(col)
        idx = feature_meta[feature_meta['feature'] == col].index
        feature_meta.loc[idx, 'b_j'] = PKT_MAX
        feature_meta.loc[idx, 'delta_f'] = PKT_MAX
        feature_meta.loc[idx, 'bound_source'] = f'Corrected: count not binary; pkt_max={PKT_MAX:.2e}'

if needs_update:
    feature_meta.to_csv(RESULTS_DIR / 'feature_meta.csv', index=False)
    print(f'UPDATED feature_meta.csv for: {needs_update}')
else:
    print('No updates needed — flag indicators are binary as expected.')

## 9. Saturation diagnostic

For each feature, computes `Δf_j / (ε × w_j)` (JSD/KL saturation ratio) and `Δf_j / (ε × σ_j)` (W1 predicted asymptote) at each ε. Planning only — does not feed the DP release.

In [ ]:
DP_EPSILONS = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

sat_rows = []
for _, row in feature_meta.iterrows():
    feat    = row['feature']
    delta_f = row['delta_f']
    vals    = analysis[feat].values.astype(float)
    sigma_j = max(vals.std(), 1e-10)
    w_j     = max((vals.max() - vals.min()) / N_BINS, 1e-10)

    for eps in DP_EPSILONS:
        ratio_jsd = delta_f / (eps * w_j)
        ratio_w1  = delta_f / (eps * sigma_j)
        if ratio_jsd < 1:
            jsd_band = 'negligible'
        elif ratio_jsd <= 100:
            jsd_band = 'INFORMATIVE'
        else:
            jsd_band = 'plateau'
        sat_rows.append({
            'feature':    feat,
            'category':   row['category'],
            'epsilon':    eps,
            'delta_f':    delta_f,
            'w_j':        w_j,
            'sigma_j':    sigma_j,
            'ratio_jsd':  ratio_jsd,
            'ratio_w1':   ratio_w1,
            'jsd_band':   jsd_band,
        })

sat_df = pd.DataFrame(sat_rows)

# Summary: features with at least one ε in the informative band
informative = sat_df[sat_df['jsd_band'] == 'INFORMATIVE'][['feature','category','epsilon','ratio_jsd']]
print(f'Features with at least one ε in informative band (1–100):')
if len(informative) == 0:
    print('  None — the entire sweep is plateau-dominated by construction.')
    print('  This is the expected finding per §5.4 of the revised proposal.')
else:
    print(informative.to_string(index=False))

print(f'\nPlateau at ALL ε (will show flat JSD curves):')
all_plateau = sat_df.groupby('feature')['jsd_band'].apply(lambda x: (x == 'plateau').all())
plateau_features = all_plateau[all_plateau].index.tolist()
print(f'  {len(plateau_features)} / {d} features → {len(plateau_features)/d*100:.0f}% of features')

In [ ]:
# Saturation diagnostic heatmap: feature × ε, colour = log10(ratio_jsd)
pivot = sat_df.pivot(index='feature', columns='epsilon', values='ratio_jsd')
# Sort by Gini importance
gini_order = gini_df['feature'].tolist()
pivot = pivot.reindex([f for f in gini_order if f in pivot.index])

fig, ax = plt.subplots(figsize=(10, max(8, d // 5)))
log_pivot = np.log10(pivot.clip(lower=1e-3))
sns.heatmap(log_pivot, ax=ax, cmap='RdYlGn_r', center=2,  # log10(100)=2 is the plateau boundary
            xticklabels=[str(e) for e in DP_EPSILONS],
            yticklabels=pivot.index, linewidths=0.2)
ax.set_title('Saturation diagnostic: log₁₀(Δf / (ε·w_j))\nGreen=informative, Red=plateau (>100), centre=boundary')
ax.set_xlabel('ε')
ax.set_ylabel('Feature (Gini-sorted)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_05_saturation_diagnostic.png', dpi=150)
plt.show()

## 10. Gate checklist

All assertions must pass before running NB2.

In [ ]:
import os

checks = [
    ((DATA_DIR / 'analysis.parquet').exists(),
     'analysis.parquet exists'),
    ((RESULTS_DIR / 'feature_meta.csv').exists(),
     'feature_meta.csv exists'),
    ((RESULTS_DIR / 'gini_importance.csv').exists(),
     'gini_importance.csv exists'),
    (len(feature_meta) == d,
     f'feature_meta has {d} rows (one per retained feature)'),
    (feature_meta['delta_f'].gt(0).all(),
     'All delta_f > 0 (no degenerate bounds)'),
    (len(gini_df) == d,
     f'gini_importance has {d} rows'),
    (60 <= d <= 78,
     f'd={d} is in expected range 60–78'),
    (pd.read_parquet(DATA_DIR / 'analysis.parquet').shape[1] == d + 1,
     'analysis.parquet has d+1 columns (d features + Label)'),
]

all_passed = True
for passed, msg in checks:
    icon = '✓' if passed else '✗'
    print(f'  {icon}  {msg}')
    if not passed:
        all_passed = False

print()
if all_passed:
    print('All gates PASSED. Ready to run NB2.')
else:
    print('Some gates FAILED. Fix the issues above before running NB2.')
    raise AssertionError('Gate check failed — see output above.')